# 4.2 Creating a RunPod Container

Prior to running this file, a RUNPOD API KEY is needed.
An ssh key is also needed. See the runpod-ssh-how-to.docx file in this same directory for instructions and an example on how to do so.

## Configuration

In [2]:
import os
import time
import runpod
import paramiko  # for SSH
from pathlib import Path

# ───────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]  # set in your environment
runpod.api_key = RUNPOD_API_KEY

In [3]:
import runpod
print(runpod.get_gpus())

[{'id': 'AMD Instinct MI300X OAM', 'displayName': 'MI300X', 'memoryInGb': 192}, {'id': 'NVIDIA A100 80GB PCIe', 'displayName': 'A100 PCIe', 'memoryInGb': 80}, {'id': 'NVIDIA A100-SXM4-80GB', 'displayName': 'A100 SXM', 'memoryInGb': 80}, {'id': 'NVIDIA A30', 'displayName': 'A30', 'memoryInGb': 24}, {'id': 'NVIDIA A40', 'displayName': 'A40', 'memoryInGb': 48}, {'id': 'NVIDIA B200', 'displayName': 'B200', 'memoryInGb': 180}, {'id': 'NVIDIA GeForce RTX 3070', 'displayName': 'RTX 3070', 'memoryInGb': 8}, {'id': 'NVIDIA GeForce RTX 3080', 'displayName': 'RTX 3080', 'memoryInGb': 10}, {'id': 'NVIDIA GeForce RTX 3080 Ti', 'displayName': 'RTX 3080 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 3090', 'displayName': 'RTX 3090', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 3090 Ti', 'displayName': 'RTX 3090 Ti', 'memoryInGb': 24}, {'id': 'NVIDIA GeForce RTX 4070 Ti', 'displayName': 'RTX 4070 Ti', 'memoryInGb': 12}, {'id': 'NVIDIA GeForce RTX 4080', 'displayName': 'RTX 4080', 'memoryInGb': 16

In [4]:
# Inteligent GPU type selection based on model size using runpod.get_gpus() and the memoryInGb field
# def select_gpu_type(model_name):
#     gpus = runpod.get_gpus()
#     gpu_map = {gpu['displayName']: gpu['id'] for gpu in gpus if gpu['isAvailable']}
    
#     if "70b" in model_name or "gemma-2-70b" in model_name:
#         return gpu_map.get("NVIDIA A100 80GB") or gpu_map.get("NVIDIA A100 40GB")
#     elif "13b" in model_name or "gemma-2-13b" in model_name:
#         return gpu_map.get("NVIDIA RTX A6000")
#     else:
#         return gpu_map.get("NVIDIA GeForce RTX 4090")

In [6]:
POD_NAME       = "lm-eval-pod-test"
IMAGE_NAME     = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
# GPU_TYPE       = "NVIDIA GeForce RTX 4090"
# GPU_TYPE       = "NVIDIA H200"
GPU_TYPE       = "NVIDIA A40"
RESULTS_FILE   = "/workspace/results.json"     # inside pod
LOCAL_RESULTS  = Path("results.json")          # where to store results locally

## Pod Creation

In [7]:
# ───────────────────────────────────────────────
# 1. Create the pod
# ───────────────────────────────────────────────
pod = runpod.create_pod(
    name=POD_NAME,
    image_name=IMAGE_NAME,
    gpu_type_id=GPU_TYPE,
    gpu_count=1,
    container_disk_in_gb=200,
    volume_in_gb=0,
    min_vcpu_count=4,
    min_memory_in_gb=16,
    ports="22/tcp,11434/http",  # expose SSH and Ollama
    env={
        "OLLAMA_HOST": "0.0.0.0",
        "PYTHONUNBUFFERED": "1"
    },
    support_public_ip=True,
    start_ssh=True
)

pod_id = pod["id"]
print(f"Created pod: {pod_id}")

raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'da2aevsmsa1e23', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'ysysvhe2p7ub', 'machine': {'podHostId': 'da2aevsmsa1e23-64411809'}}}}
Created pod: da2aevsmsa1e23


In [8]:
details = runpod.get_pod(pod_id)
print("DEBUG details:", details)


DEBUG details: {'id': 'da2aevsmsa1e23', 'containerDiskInGb': 200, 'costPerHr': 0.4, 'desiredStatus': 'RUNNING', 'dockerArgs': None, 'dockerId': None, 'env': ['OLLAMA_HOST=0.0.0.0', 'PYTHONUNBUFFERED=1', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'gpuCount': 1, 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'lastStatusChange': 'Rented by User: Thu Sep 25 2025 04:46:34 GMT+0000 (Coordinated Universal Time)', 'machineId': 'ysysvhe2p7ub', 'memoryInGb': 50, 'name': 'lm-eval-pod-test', 'podType': 'RESERVED', 'port': None, 'ports': '22/tcp,11434/http', 'uptimeSeconds': 0, 'vcpuCount': 9, 'volumeInGb': 0, 'volumeMountPath': '/runpod-volume', 'runtime': {'ports': [{'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 11434, 'publicPort': 60831, 'type': 'http'}, {'ip': '100.65.24.9', 'isIpPublic': False, 'privatePort': 19123, 'publicPort': 60832, 'type': 'http'}, {'ip': '69.30.

## Wait for pod to become RUNNING and get SSH endpoint

In [11]:
# ───────────────────────────────────────────────
# 2. Wait for pod to become RUNNING and get SSH endpoint
# ───────────────────────────────────────────────
import time

ssh_host = None
ssh_port = None

print("Waiting for pod to become RUNNING and for SSH endpoint to appear...")
while True:
    details = runpod.get_pod(pod_id)
    status = details.get("desiredStatus")
    print(f"  Current status: {status}")

    # If the pod is running, check for a public SSH port
    if status == "RUNNING":
        runtime = details.get("runtime")
        if runtime and runtime.get("ports"):
            for p in runtime["ports"]:
                if p["type"] == "tcp" and p["privatePort"] == 22 and p["isIpPublic"]:
                    ssh_host = p["ip"]
                    ssh_port = p["publicPort"]
                    break
            if ssh_host:
                break   # Exit the loop once we have the SSH endpoint

    time.sleep(10)

print(f"Pod is RUNNING with SSH ready at {ssh_host}:{ssh_port}")




Waiting for pod to become RUNNING and for SSH endpoint to appear...
  Current status: RUNNING
Pod is RUNNING with SSH ready at 69.30.85.132:22198


## Connect to the pod

In [5]:
# ───────────────────────────────────────────────
# 3A Connect to the pod
# ───────────────────────────────────────────────
import os
import time
import paramiko
import runpod

# Connect automatically with Paramiko
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())  # auto-accept host fingerprint
ssh.connect(ssh_host, port=ssh_port, username="root", key_filename=ssh_key_path)
print("Connected to pod.")

# Connect with your key (no passphrase)
ssh_key_path = os.path.expanduser("~/.ssh/id_ed25519")
pkey = paramiko.Ed25519Key.from_private_key_file(ssh_key_path)
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(ssh_host, port=ssh_port, username="root", pkey=pkey)
print("SSH connection established.")

Connected to pod.
SSH connection established.


## Provision the pod

In [ ]:
print("Starting provisioning...")

# --- Install prerequisites and Ollama ---
# base_commands = [
#     "apt-get update && apt-get install -y curl git python3-pip lshw",
#     "curl -fsSL https://ollama.com/install.sh | sh",
#     # start Ollama in background and detach so Paramiko doesn't hang
#     "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
#     "sleep 10"   # give the server time to start
# ]

base_commands = [
    # "apt-get update && apt-get install -y curl git python3-pip lshw",
    "apt update && apt install lshw -y",             # optional
    "curl -fsSL https://ollama.com/install.sh | sh", # install Ollama binary
    "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
    "sleep 10",
]

for cmd in base_commands:
    print(f"Running: {cmd}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    print(stdout.read().decode())
    err = stderr.read().decode()
    if err: print("ERROR:", err)

# --- Pull the model ---
model_name = "llama3.2:1b"
print(f"Pulling model {model_name} ...")
stdin, stdout, stderr = ssh.exec_command(f"ollama pull {model_name}")
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Install evaluation harness ---
stdin, stdout, stderr = ssh.exec_command(
    "pip install lm_eval lm_eval[api] ollama==0.3.3"
)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)

# --- Check until the model appears in 'ollama list' ---
print(f"Waiting for model '{model_name}' to show up in `ollama list`...")
while True:
    stdin, stdout, stderr = ssh.exec_command("ollama list")
    output = stdout.read().decode()
    if model_name.split(":")[0] in output:
        print("Model is ready.")
        break
    print("  Not ready yet, retrying in 10 seconds...")
    time.sleep(10)

print("Provisioning complete.")


Starting provisioning...
Running: apt update && apt install lshw -y
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Reading package lists...
Building dependency tree...
Reading state information...
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists...
Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.

ERROR: 




Running: curl -fsSL https://ollama.com/install.sh | sh

ERROR: >>> Cleaning up old version at /usr/local/lib/ollama
>>> Inst

In [9]:
# Pushing the yaml file to the pod
yaml_content = r'''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  until: []  # No early stopping for thinking models
  max_gen_toks: 10000  # High limit for reasoning traces
  temperature: 0.0  # Deterministic for evaluation

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true'''

remote_file = "/root/sysengbench.yaml"
command = f"cat > {remote_file} <<'EOF'\n{yaml_content}\nEOF"

stdin, stdout, stderr = ssh.exec_command(command)
print(stdout.read().decode(), stderr.read().decode())


## Run the eval on the pod

In [12]:
model = "llama3.2:1b"
base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
include_path = "./"
tasks = "sysengbench"
output_dir = "output/sysengbench/"
log_samples = True
batch_size = "auto"
temperature = 0.0
apply_chat_template = True

# Construct the benchmark command dynamically
log_samples_flag = "--log_samples" if log_samples else ""
apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

lm_eval_cmd = f"""
lm_eval \
  --model local-chat-completions \
  --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
  --include_path {include_path} \
  --tasks {tasks} \
  --output {output_dir} \
  {log_samples_flag} \
  --num_fewshot 0 \
  --batch_size {batch_size} \
  --limit 10 \
  --gen_kwargs temperature={temperature} \
  {apply_template_flag}
"""

stdin, stdout, stderr = ssh.exec_command(lm_eval_cmd)
print(stdout.read().decode())
err = stderr.read().decode()
if err: print("ERROR:", err)


local-chat-completions (model=llama3.2:1b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1), gen_kwargs: (temperature=0.0), limit: 10.0, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |  0.9|±  |   0.1|


ERROR: 2025-09-25:05:30:13 INFO     [__main__:348] Including path: ./
2025-09-25:05:30:16 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-09-25:05:30:16 INFO     [__main__:446] Selected Tasks: ['sysengbench']
2025-09-25:05:30:16 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-09-25:05:30:16 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will up

## Recover the output files from evals on the pod

In [ ]:
# confirming files are indeed placed in root.
stdin, stdout, stderr = ssh.exec_command(
    "find /root -type f \\( -name 'results_*.json' -o -name 'samples_*.jsonl' \\) 2>/dev/null"
)
matches = stdout.read().decode().strip().splitlines()
print("Found files:", matches)


Found files: ['/root/output/sysengbench/llama3.2__1b/results_2025-09-25T05-30-23.489092.json', '/root/output/sysengbench/llama3.2__1b/samples_sysengbench_2025-09-25T05-30-23.489092.jsonl']


In [15]:
import os
import stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    os.makedirs(local_dir, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir,  entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)  # recurse into subdir
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")


In [ ]:
# permissions check
for path in ["/root", "/root/output", "/root/output/sysengbench"]:
    stdin, stdout, stderr = ssh.exec_command(f"ls -ld {path}")
    print(path, stdout.read().decode(), stderr.read().decode())


/root drwx------ 1 root root 143 Sep 25 05:30 /root
 
/root/output drwxr-xr-x 3 root root 33 Sep 25 05:30 /root/output
 
/root/output/sysengbench drwxr-xr-x 3 root root 34 Sep 25 05:30 /root/output/sysengbench
 


In [21]:
remote_dir = "/root/output"
local_dir  = "C:/Users/rabel/Desktop/dissertation-outputs/runpod_results"

import os, stat

def sftp_get_dir(sftp, remote_dir, local_dir):
    """
    Recursively download remote_dir from the pod to local_dir,
    preserving the folder structure.
    """
    try:
        sftp.chdir(remote_dir)
    except IOError:
        print(f"Remote directory not found: {remote_dir}")
        return
    os.makedirs(local_dir, exist_ok=True)

    for entry in sftp.listdir_attr(remote_dir):
        remote_path = os.path.join(remote_dir, entry.filename)
        local_path  = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir(sftp, remote_path, local_path)
        else:
            sftp.get(remote_path, local_path)
            print(f"Downloaded {remote_path} -> {local_path}")

# ---- usage ----
sftp = ssh.open_sftp()
sftp_get_dir(sftp, remote_dir, local_dir)
sftp.close()

print(f"All files downloaded to: {local_dir}")


Remote directory not found: /root/output\sysengbench
All files downloaded to: C:/Users/rabel/Desktop/dissertation-outputs/runpod_results


In [23]:
import posixpath  # <-- always uses forward slashes
import os, stat

def sftp_get_dir_fixed(sftp, remote_dir, local_dir, depth=0):
    indent = "  " * depth
    print(f"{indent}Entering remote: {remote_dir}")

    try:
        sftp.chdir(remote_dir)
    except IOError as e:
        print(f"{indent}❌ Cannot access {remote_dir}: {e}")
        return

    os.makedirs(local_dir, exist_ok=True)
    print(f"{indent}Local target: {local_dir}")

    entries = sftp.listdir_attr(remote_dir)
    if not entries:
        print(f"{indent}(empty directory)")
        return

    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)  # ✅ POSIX join
        local_path  = os.path.join(local_dir, entry.filename)      # local join is fine
        print(f"{indent}- {entry.filename} "
              f"{'DIR' if stat.S_ISDIR(entry.st_mode) else 'FILE'}")

        if stat.S_ISDIR(entry.st_mode):
            sftp_get_dir_fixed(sftp, remote_path, local_path, depth + 1)
        else:
            try:
                sftp.get(remote_path, local_path)
                print(f"{indent}  ✅ Downloaded to {local_path}")
            except Exception as e:
                print(f"{indent}  ❌ Failed to download {remote_path}: {e}")


In [24]:
remote_dir = "/root/output"
local_dir  = r"C:\Users\rabel\Desktop\dissertation-outputs\runpod_results"

sftp = ssh.open_sftp()
sftp_get_dir_fixed(sftp, remote_dir, local_dir)
sftp.close()


Entering remote: /root/output
Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results
- sysengbench DIR
  Entering remote: /root/output/sysengbench
  Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench
  - llama3.2__1b DIR
    Entering remote: /root/output/sysengbench/llama3.2__1b
    Local target: C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b
    - results_2025-09-25T05-30-23.489092.json FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\results_2025-09-25T05-30-23.489092.json
    - samples_sysengbench_2025-09-25T05-30-23.489092.jsonl FILE
      ✅ Downloaded to C:\Users\rabel\Desktop\dissertation-outputs\runpod_results\sysengbench\llama3.2__1b\samples_sysengbench_2025-09-25T05-30-23.489092.jsonl


## Terminate pod

In [25]:
# ───────────────────────────────────────────────
# 6. Terminate pod
# ───────────────────────────────────────────────
runpod.terminate_pod(pod_id)
print(f"Pod {pod_id} terminated.")

ssh.close()

Pod da2aevsmsa1e23 terminated.


## recovering/ reconnecting to a pod

In [3]:
import os, time, runpod

runpod.api_key = os.environ["RUNPOD_API_KEY"]

def reconnect_by_name(pod_name):
    """
    Find an existing RUNNING pod by its name and return its id, public IP and public SSH port.
    """
    pods = runpod.get_pods()
    for p in pods:
        if p["name"] == pod_name and p["desiredStatus"] == "RUNNING":
            details = runpod.get_pod(p["id"])
            runtime = details.get("runtime", {})
            if runtime and runtime.get("ports"):
                for port in runtime["ports"]:
                    if port["type"] == "tcp" and port["privatePort"] == 22 and port["isIpPublic"]:
                        return {
                            "id": details["id"],
                            "host": port["ip"],
                            "port": port["publicPort"]
                        }
    raise RuntimeError(f"No running pod named '{pod_name}' found.")

# Example usage:
pod_name = "lm-eval-pod-test"   # <-- the name you used in create_pod
conn = reconnect_by_name(pod_name)
pod_id, ssh_host, ssh_port = conn["id"], conn["host"], conn["port"]
print(f"Reconnected to {pod_name} -> {ssh_host}:{ssh_port}")


Reconnected to lm-eval-pod-test -> 69.30.85.132:22198
